# Exploratory Data Analysis (EDA)

This notebook explores the solar power generation dataset to understand its structure, distributions, and key patterns.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Load all 5-Min actual power data files
data_dir = '../data/raw'
csv_files = glob.glob(os.path.join(data_dir, 'Actual_*_5_Min.csv'))
print(f"Found {len(csv_files)} 5-Min power data files\n")

# Load and combine all files
dfs = []
for f in sorted(csv_files)[:10]:  # Load first 10 for manageable EDA
    df = pd.read_csv(f)
    dfs.append(df)
    print(f"  {os.path.basename(f)}: {len(df)} rows")

df = pd.concat(dfs, ignore_index=True)
print(f"\nCombined dataset: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Display shape, columns, dtypes
print("="*50)
print("DATASET OVERVIEW")
print("="*50)
print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")

print("\nColumn Names and Data Types:")
print("-"*40)
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

print("\nFirst 10 rows:")
df.head(10)

## 2. Target Analysis (Regression)

Since `Power(MW)` is a continuous variable, this is a **regression** problem.
We analyze the target distribution and summary statistics.

In [ ]:
# Target Analysis for Regression
target_col = 'Power(MW)'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of target
axes[0].hist(df[target_col], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Power (MW)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Target Distribution: Power(MW) - Histogram')

# Box plot of target
axes[1].boxplot(df[target_col].dropna())
axes[1].set_ylabel('Power (MW)')
axes[1].set_title('Target Distribution: Power(MW) - Box Plot')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
print("\nTarget Variable Statistics:")
print("-"*40)
print(f"  Mean:   {df[target_col].mean():.4f} MW")
print(f"  Std:    {df[target_col].std():.4f} MW")
print(f"  Min:    {df[target_col].min():.4f} MW")
print(f"  Max:    {df[target_col].max():.4f} MW")
print(f"  Median: {df[target_col].median():.4f} MW")

## 3. Missing Values

Visualize null counts per column using a heatmap.

In [ ]:
# Missing Values Analysis
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

print("Missing Values per Column:")
print("-"*40)
for col in df.columns:
    print(f"  {col}: {missing_counts[col]} ({missing_pct[col]:.2f}%)")

# Heatmap of missing values
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar_kws={'label': 'Null Present'})
plt.title('Missing Values Heatmap')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.tight_layout()
plt.savefig('missing_values_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Distributions

Show histograms for all numeric features in a 3x3 subplot grid.

In [ ]:
# Feature Distributions
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

n_features = len(numeric_cols)
n_rows = 3
n_cols = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 12))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'Distribution: {col}')

# Hide unused subplots
for idx in range(len(numeric_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Matrix

Compute and visualize correlations between numeric features.

In [ ]:
# Correlation Matrix
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrelation with Target (Power):")
print("-"*40)
if 'Power(MW)' in corr.columns:
    target_corr = corr['Power(MW)'].drop('Power(MW)').sort_values(ascending=False)
    for feat, val in target_corr.items():
        print(f"  {feat}: {val:.4f}")

## 6. Features vs Target

For regression: show scatter plots of the 3 strongest features correlated with the target.

In [ ]:
# Features vs Target (Regression - Scatter Plots)
# Get correlation with target
if 'Power(MW)' in numeric_cols:
    feature_cols = [c for c in numeric_cols if c != 'Power(MW)']
    corrs = df[feature_cols].corrwith(df['Power(MW)']).abs().sort_values(ascending=False)
    top_features = corrs.head(3).index.tolist()
    print(f"Top 3 features correlated with Power(MW): {top_features}")
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    for idx, feat in enumerate(top_features):
        axes[idx].scatter(df[feat], df['Power(MW)'], alpha=0.3, s=5)
        axes[idx].set_xlabel(feat)
        axes[idx].set_ylabel('Power (MW)')
        axes[idx].set_title(f'{feat} vs Power(MW)\n(r={corrs[feat]:.4f})')
    
    plt.tight_layout()
    plt.savefig('features_vs_target.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Power(MW) column not found in numeric columns")

## 7. Key Findings

### Summary of Patterns Discovered

- **Power Output Range**: The target variable `Power(MW)` ranges from 0 to ~172 MW, with most values clustered near 0 (nighttime/zero generation periods).

- **Zero-Dominant Distribution**: The target histogram shows a strong peak at zero, indicating that solar power generation frequently hits minimum output levels during night hours or cloudy conditions.

- **No Missing Values**: The dataset appears complete with no null values in either column, ensuring clean analysis without imputation.

- **Potential Features**: Since the data contains time information (`LocalTime`), hour-of-day and time-based features could be engineered to capture solar generation patterns.

- **Strong Temporal Pattern Expected**: Solar power generation follows diurnal cycles — correlation analysis with engineered time features should reveal strong predictable patterns.